# Feedforward skip edges

This notebook explains how `edge2torch` handles skip edges in the
`feedforward` backend.

`edge2torch` builds sparse neural networks from edgelists of named
nodes. In the feedforward backend, the compiled model is evaluated
strictly layer by layer. Some valid feedforward graphs, however,
contain edges that skip over one or more intermediate layers.

To represent those skip edges in a strictly layer-wise execution plan,
`edge2torch` expands them internally into pseudo nodes.

Pseudo nodes are:

- internal compiler-generated nodes
- used only during compilation and execution
- inserted only by the `feedforward` backend
- hidden from user-facing node attribution outputs

The goal is to let users provide natural architecture edgelists while
keeping the compiled feedforward model compatible with regular batched
tensor operations.


## What problem do pseudo nodes solve?

In a strictly layer-wise feedforward model, each computation block maps
from one layer to the next.

A graph such as this is naturally layer-wise:

`feature_1 -> hidden_1 -> hidden_2 -> prediction`

But a direct skip edge such as this crosses multiple layers:

`feature_1 -> prediction`

The `feedforward` backend handles this by representing the skip edge
internally as an adjacent layer-to-layer path through pseudo nodes.
For example, the skip edge may be expanded internally as:

`feature_1 -> pseudo_node_1 -> pseudo_node_2 -> prediction`

This keeps the compiled computation regular: every compiled block maps
from one layer to the next.

This layer-wise execution pattern maps naturally to PyTorch tensor
operations and GPU execution. Pseudo-node expansion may increase the
internal graph size for long skip edges, so it should be understood as
a compilation strategy for regular batched execution rather than a
guarantee of faster runtime for every graph.

The figure below shows the computational strategy behind the pseudo nodes.

![Pseudo nodes explained](../figures/pseudo_nodes_explained.svg)

**Figure 1.** For a skip edge that spans multiple layers, `edge2torch`
inserts one or more pseudo nodes into the skipped layers. The original skip
edge is then represented as a chain of adjacent layer-to-layer connections.
During forward propagation, each pseudo node passes through the activation it
receives from the previous layer: its output activation is overwritten by its
input activation. This lets the pseudo-node path carry the skip-edge activation
through the layer-wise execution plan, preserving the original graph
connectivity while allowing the compiled feedforward model to run as regular
batched layer operations.

> **Interpretation note**
>
> Pseudo nodes are internal implementation nodes. They are not returned as
> user-facing node attributions. Attribution results are mapped back to the
> original named graph nodes, while pseudo nodes serve only as internal carriers
> along expanded skip-edge paths.

> **Backend scope**
>
> Pseudo nodes are currently used only by the `feedforward` backend. They allow
> skip edges to be represented as adjacent layer-to-layer transitions.
>
> The `recurrent` and `graphnn` backends operate directly on the graph topology
> and do not currently use pseudo nodes.